# 🗺️ CRASH LENS — Geographic Data Downloader (Colab Edition)

**Replaces:** `download-geographic-data.yml` GitHub Actions workflow

### Data Sources
| Layer | Source | Notes |
|-------|--------|-------|
| States | Census Gazetteer (`www2.census.gov`) | TIGERweb blocks cloud IPs |
| Counties | Census Gazetteer (`www2.census.gov`) | TIGERweb blocks cloud IPs |
| Places | Census Gazetteer (`www2.census.gov`) | TIGERweb blocks cloud IPs |
| Subdivisions | Census Gazetteer (`www2.census.gov`) | TIGERweb blocks cloud IPs |
| MPOs | BTS ArcGIS Online (`services.arcgis.com`) | With centroid geometry |

### Schema Compatibility
Output JSON files are **schema-compatible with TIGERweb** — missing fields
(`MTFCC`, `NAMELSAD`, `LSADC`, `FUNCSTAT`, `OBJECTID`, etc.) are backfilled
with correct values so downstream CRASH LENS processing works unchanged.

**v2.3 fixes:** BASENAME stripping for all place types, accurate LSADC for
towns/villages/boroughs/municipalities, hardcoded STATENS, DC/PR LSADC codes.

**Output:** `data/geographic/*.json`


## ⚙️ Configuration

In [ ]:
#@title Configuration { display-mode: "form" }
#@markdown ### What to download
SCOPE = "all"  #@param ["all", "states", "counties", "places", "subdivisions", "mpos", "counties-and-mpos"]

#@markdown ### Limit to single state (2-digit FIPS, blank = nationwide)
STATE_FIPS = ""  #@param {type:"string"}

#@markdown ### Skip county subdivisions (saves ~5 min)
SKIP_SUBDIVISIONS = True  #@param {type:"boolean"}

#@markdown ### Save to Google Drive
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

#@markdown ---

import os, json, time, datetime, io, csv, zipfile

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = "/content/drive/MyDrive/crash-lens/data/geographic"
else:
    OUTPUT_DIR = "/content/data/geographic"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\u2713 Output directory: {OUTPUT_DIR}")
print(f"  Scope: {SCOPE}")
print(f"  State filter: {STATE_FIPS or 'nationwide'}")
print(f"  Skip subdivisions: {SKIP_SUBDIVISIONS}")


## 📦 Core Download Engine

In [ ]:
import requests, csv, io, zipfile, json, os, time, datetime
from collections import OrderedDict

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "CrashLens-GeoDownloader/2.3"})

GAZETTEER_BASE = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer"
GAZETTEER_YEAR = "2024"

# ============================================================================
# Reference lookups for TIGERweb schema backfill
# ============================================================================

# ── STATENS: ANSI state numeric codes ──
# Source: Census ANSI FIPS codes (not in Gazetteer state file)
STATENS_LOOKUP = {
    "01": "01779775", "02": "01785533", "04": "01779777", "05": "00068085",
    "06": "01779778", "08": "01779779", "09": "01779780", "10": "01779781",
    "11": "01702382", "12": "00294478", "13": "01705317", "15": "01779782",
    "16": "01779783", "17": "01779784", "18": "00448508", "19": "01779785",
    "20": "00481813", "21": "01779786", "22": "01629543", "23": "01779787",
    "24": "01714934", "25": "00606926", "26": "01779789", "27": "00662849",
    "28": "01779790", "29": "01779791", "30": "00767982", "31": "01779792",
    "32": "01779793", "33": "01779794", "34": "01779795", "35": "00897535",
    "36": "01779796", "37": "01027616", "38": "01779797", "39": "01085497",
    "40": "01102857", "41": "01155107", "42": "01779798", "44": "01219835",
    "45": "01779799", "46": "01785534", "47": "01325873", "48": "01779801",
    "49": "01455989", "50": "01779802", "51": "01779803", "53": "01779804",
    "54": "01779805", "55": "01779806", "56": "01779807", "72": "01779808",
}

# ── State LSADC codes ──
STATE_LSADC = {
    "11": "03",   # DC — Federal District
    "72": "72",   # PR — Commonwealth
    # All 50 states = "00"
}

# ── County LSADC — derived from name suffix ──
COUNTY_SUFFIX_TO_LSADC = {
    " County": "06",
    " Parish": "15",
    " Borough": "04",
    " Census Area": "13",
    " Municipality": "12",
    " city": "25",            # VA independent cities (lowercase)
    " City and Borough": "04",
    " Municipio": "14",       # PR
    " District": "07",        # DC (not actually used — DC has no counties)
}

# ── Place LSADC — derived from name suffix ──
# TIGERweb codes: 25=city, 43=town, 47=village, 49=borough,
#   53=municipality, 55=plantation, 57=CDP
#   62=comunidad(PR), 59=zona urbana(PR)
PLACE_SUFFIX_TO_LSADC = {
    " city": "25",
    " town": "43",
    " village": "47",
    " borough": "49",
    " municipality": "53",
    " plantation": "55",
    " CDP": "57",
    " comunidad": "62",
    " zona urbana": "59",
    # Consolidated/unified government — treat as city
    " metro township": "43",
    " corporation": "25",
    " urban county": "25",
}

# ── Place BASENAME stripping ──
# Ordered longest-first so " city (balance)" matches before " city"
PLACE_SUFFIXES_ORDERED = [
    " unified government (balance)",
    " consolidated government (balance)",
    " metro government (balance)",
    " metropolitan government (balance)",
    " city (balance)",
    " unified government",
    " consolidated government",
    " metro government",
    " metropolitan government",
    " metro township",
    " urban county",
    " zona urbana",
    " corporation",
    " municipality",
    " comunidad",
    " plantation",
    " borough",
    " village",
    " town",
    " city",
    " CDP",
]

# ── MTFCC codes ──
MTFCC = {
    "states": "G4000",
    "counties": "G4020",
    "places_incorporated": "G4110",
    "places_cdp": "G4210",
    "subdivisions": "G4040",
}


def _strip_place_suffix(name):
    """Strip place-type suffix to get BASENAME. Returns (basename, suffix)."""
    for suffix in PLACE_SUFFIXES_ORDERED:
        if name.endswith(suffix):
            return name[:-len(suffix)].strip(), suffix
    return name, ""


def _place_lsadc(name, funcstat):
    """Derive LSADC from place name suffix."""
    if funcstat == "S":
        return "57"  # CDP
    for suffix, code in PLACE_SUFFIX_TO_LSADC.items():
        if name.endswith(suffix):
            return code
    return "25"  # Default to city for incorporated places


def _strip_county_suffix(name):
    """Strip county-type suffix to get BASENAME."""
    for suffix in COUNTY_SUFFIX_TO_LSADC:
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return name


def _county_lsadc(name):
    """Derive LSADC from county name suffix."""
    for suffix, code in COUNTY_SUFFIX_TO_LSADC.items():
        if name.endswith(suffix):
            return code
    return "06"


# ============================================================================
# Layer config
# ============================================================================
LAYER_CONFIG = {
    "states": {
        "gaz_file": f"{GAZETTEER_YEAR}_Gaz_state_national.zip",
        "filename": "us_states.json",
        "desc": "States",
        "geoid_parts": {"STATE": (0, 2)},
    },
    "counties": {
        "gaz_file": f"{GAZETTEER_YEAR}_Gaz_counties_national.zip",
        "filename": "us_counties.json",
        "desc": "Counties",
        "geoid_parts": {"STATE": (0, 2), "COUNTY": (2, 5)},
    },
    "places": {
        "gaz_file": f"{GAZETTEER_YEAR}_Gaz_place_national.zip",
        "filename": "us_places.json",
        "desc": "Incorporated Places",
        "geoid_parts": {"STATE": (0, 2), "PLACE": (2, 7)},
    },
    "subdivisions": {
        "gaz_file": f"{GAZETTEER_YEAR}_Gaz_cousubs_national.zip",
        "filename": "us_county_subdivisions.json",
        "desc": "County Subdivisions",
        "geoid_parts": {"STATE": (0, 2), "COUNTY": (2, 5), "COUSUB": (5, 10)},
    },
}


# ============================================================================
# Download & parse
# ============================================================================
def download_gazetteer(layer_key, state_fips=None):
    """Download Census Gazetteer ZIP and parse to TIGERweb-compatible schema."""
    cfg = LAYER_CONFIG[layer_key]
    url = f"{GAZETTEER_BASE}/{GAZETTEER_YEAR}_Gazetteer/{cfg['gaz_file']}"

    print(f"  Downloading {url}")
    resp = SESSION.get(url, timeout=120)
    resp.raise_for_status()

    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    txt_name = [n for n in zf.namelist() if n.endswith('.txt')][0]
    raw = zf.read(txt_name).decode('utf-8', errors='replace')
    reader = csv.DictReader(io.StringIO(raw), delimiter='\t')

    records = []
    obj_id = 0

    for row in reader:
        row = {k.strip(): v.strip() if v else '' for k, v in row.items()}
        geoid = row.get('GEOID', '')

        if state_fips and len(geoid) >= 2 and geoid[:2] != state_fips:
            continue

        obj_id += 1
        rec = OrderedDict()

        # ── Core identifiers ──
        rec['GEOID'] = geoid
        for field, (start, end) in cfg['geoid_parts'].items():
            rec[field] = geoid[start:end] if len(geoid) >= end else ''

        name = row.get('NAME', '')
        rec['NAME'] = name

        # ── Layer-specific backfill ──
        if layer_key == "states":
            state_code = geoid[:2]
            rec['MTFCC'] = MTFCC["states"]
            rec['BASENAME'] = name
            rec['NAMELSAD'] = name
            rec['LSADC'] = STATE_LSADC.get(state_code, "00")
            rec['FUNCSTAT'] = "A"
            rec['STATENS'] = STATENS_LOOKUP.get(state_code, '')

        elif layer_key == "counties":
            rec['MTFCC'] = MTFCC["counties"]
            rec['BASENAME'] = _strip_county_suffix(name)
            rec['NAMELSAD'] = name
            rec['LSADC'] = _county_lsadc(name)
            rec['FUNCSTAT'] = row.get('FUNCSTAT', 'A')
            rec['COUNTYNS'] = row.get('ANSICODE', '')

        elif layer_key == "places":
            funcstat = row.get('FUNCSTAT', 'A')
            rec['FUNCSTAT'] = funcstat
            rec['MTFCC'] = MTFCC["places_cdp"] if funcstat == 'S' else MTFCC["places_incorporated"]
            basename, suffix = _strip_place_suffix(name)
            rec['BASENAME'] = basename
            rec['NAMELSAD'] = name
            rec['LSADC'] = _place_lsadc(name, funcstat)
            rec['PLACENS'] = row.get('ANSICODE', '')
            rec['PLACECC'] = ""

        elif layer_key == "subdivisions":
            rec['MTFCC'] = MTFCC["subdivisions"]
            rec['BASENAME'] = name
            rec['NAMELSAD'] = name
            rec['LSADC'] = "22"
            rec['FUNCSTAT'] = row.get('FUNCSTAT', 'A')
            rec['COUSUBNS'] = row.get('ANSICODE', '')

        # ── Common fields ──
        rec['USPS'] = row.get('USPS', '')
        rec['ANSICODE'] = row.get('ANSICODE', '')
        rec['OID'] = str(obj_id)
        rec['OBJECTID'] = obj_id
        rec['AREALAND'] = int(row['ALAND']) if row.get('ALAND') else 0
        rec['AREAWATER'] = int(row['AWATER']) if row.get('AWATER') else 0

        lat = row.get('INTPTLAT', '').replace('+', '')
        lon = row.get('INTPTLONG', row.get('INTPTLON', ''))
        rec['INTPTLAT'] = lat
        rec['INTPTLON'] = lon
        rec['CENTLAT'] = lat
        rec['CENTLON'] = lon

        records.append(rec)

    if state_fips:
        print(f"  Filtered to state FIPS {state_fips}: {len(records)} records")
    else:
        print(f"  Parsed {len(records)} records")

    return records


def download_mpos(state_fips=None):
    """Download MPOs from BTS ArcGIS Online with geometry for centroid computation."""
    base_url = ("https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/"
                "NTAD_Metropolitan_Planning_Organizations/FeatureServer/0/query")

    all_features = []
    offset = 0
    page_size = 200

    print(f"  Fetching MPOs with geometry (for centroid computation)...")

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "geometryPrecision": 4,
            "outSR": "4326",
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }

        for attempt in range(1, 5):
            try:
                r = SESSION.get(base_url, params=params, timeout=120)
                r.raise_for_status()
                data = r.json()

                if "error" in data:
                    code_val = data["error"].get("code", "?")
                    msg = data["error"].get("message", "")
                    print(f"  \u2717 ArcGIS error (code {code_val}): {msg}")
                    if offset == 0:
                        print("  Falling back to attribute-only query...")
                        params["returnGeometry"] = "false"
                        params.pop("geometryPrecision", None)
                        params.pop("outSR", None)
                        r2 = SESSION.get(base_url, params=params, timeout=90)
                        data = r2.json()
                        if "features" in data:
                            all_features = data["features"]
                    return _process_mpos(all_features)

                features = data.get("features", [])
                all_features.extend(features)

                if len(features) < page_size:
                    print(f"  Fetched {len(all_features)} MPOs with geometry")
                    return _process_mpos(all_features)

                offset += page_size
                print(f"  Fetched {len(all_features)} MPOs...")
                break

            except requests.exceptions.RequestException as e:
                wait = 2 ** attempt
                print(f"  \u26a0 Request error (attempt {attempt}/4): {e}")
                if attempt < 4:
                    time.sleep(wait)
                else:
                    return _process_mpos(all_features)

    return _process_mpos(all_features)


def _compute_centroid(geometry):
    """Compute centroid from ArcGIS polygon geometry (rings)."""
    if not geometry or "rings" not in geometry:
        return None, None

    all_x, all_y = [], []
    for ring in geometry["rings"]:
        for point in ring:
            if len(point) >= 2:
                all_x.append(point[0])
                all_y.append(point[1])

    if not all_x:
        return None, None

    return round(sum(all_y) / len(all_y), 6), round(sum(all_x) / len(all_x), 6)


def _process_mpos(features):
    """Convert raw ArcGIS features to TIGERweb-compatible records with centroids."""
    records = []

    for i, feat in enumerate(features, 1):
        attrs = feat.get("attributes", {})
        geom = feat.get("geometry", None)

        rec = OrderedDict()
        rec['OBJECTID'] = attrs.get('OBJECTID', i)
        rec['MPO_ID'] = attrs.get('MPO_ID', '')
        rec['MPO_NAME'] = attrs.get('MPO_NAME', '')
        rec['NAME'] = attrs.get('MPO_NAME', '')
        rec['STATE'] = attrs.get('STATE', '')
        rec['AREA'] = attrs.get('AREA', 0)

        centlat, centlon = _compute_centroid(geom)
        rec['CENTLAT'] = str(centlat) if centlat else ''
        rec['CENTLON'] = str(centlon) if centlon else ''
        rec['INTPTLAT'] = rec['CENTLAT']
        rec['INTPTLON'] = rec['CENTLON']

        rec['GEOID'] = attrs.get('MPO_ID', str(i))
        area_sqmi = attrs.get('AREA', 0) or 0
        rec['AREALAND'] = int(area_sqmi * 2589988.11)
        rec['AREAWATER'] = 0

        records.append(rec)

    with_centroids = sum(1 for r in records if r.get('CENTLAT'))
    print(f"  Computed centroids for {with_centroids}/{len(records)} MPOs")
    return records


def save_layer(records, filename, layer_desc, output_dir):
    """Save records to JSON with metadata header."""
    output = {
        "_metadata": {
            "source": "CrashLens Geographic Data Downloader (Colab v2.3)",
            "dataSource": "Census Gazetteer + BTS/USDOT ArcGIS Online",
            "schemaCompat": "TIGERweb-backfilled",
            "gazetteerYear": GAZETTEER_YEAR,
            "downloadDate": datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "recordCount": len(records),
            "layerDescription": layer_desc
        },
        "records": records
    }
    filepath = os.path.join(output_dir, filename)
    with open(filepath, "w") as f:
        json.dump(output, f, indent=None, separators=(",", ":"))

    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  \U0001f4be Saved {filepath} ({size_mb:.1f} MB, {len(records)} records)")
    return len(records)


print("\u2713 Download engine loaded (v2.3 \u2014 full TIGERweb schema backfill)")
print(f"  Gazetteer: {GAZETTEER_BASE}/{GAZETTEER_YEAR}_Gazetteer/")
print(f"  MPOs: services.arcgis.com (BTS/USDOT NTAD, with geometry)")
print(f"  Backfill: MTFCC, NAMELSAD, LSADC (per-suffix), FUNCSTAT,")
print(f"            STATENS (hardcoded), BASENAME (stripped), OBJECTID")


## 🚀 Run Download

In [ ]:
# Build the list of layers to download
if SCOPE == "all":
    layers = ["states", "counties", "places", "mpos"]
    if not SKIP_SUBDIVISIONS:
        layers.insert(3, "subdivisions")
elif SCOPE == "counties-and-mpos":
    layers = ["counties", "mpos"]
else:
    layers = [SCOPE]

print("=" * 60)
print("  CRASH LENS \u2014 Geographic Data Downloader (Colab v2.3)")
print(f"  Date: {datetime.datetime.now(datetime.UTC).strftime('%Y-%m-%d %H:%M')} UTC")
print(f"  Scope: {SCOPE} \u2192 layers: {layers}")
print(f"  State filter: {STATE_FIPS or 'nationwide'}")
print(f"  Schema: TIGERweb-compatible (full backfill)")
print("=" * 60)

results = {}
total_records = 0
errors = []

for layer_key in layers:
    if layer_key == "mpos":
        desc = "MPOs (BTS/USDOT)"
        filename = "us_mpos.json"
    else:
        cfg = LAYER_CONFIG[layer_key]
        desc = cfg["desc"]
        filename = cfg["filename"]

    print(f"\n\u2550\u2550\u2550 Downloading {desc} \u2550\u2550\u2550\n")

    try:
        if layer_key == "mpos":
            records = download_mpos(state_fips=STATE_FIPS or None)
        else:
            records = download_gazetteer(layer_key, state_fips=STATE_FIPS or None)

        count = save_layer(records, filename, desc, OUTPUT_DIR)
        results[filename] = count
        total_records += count

    except Exception as e:
        import traceback
        print(f"  \u2717 FAILED: {e}")
        traceback.print_exc()
        errors.append((desc, str(e)))
        results[filename] = 0

# ── Summary ──
print(f"\n{'=' * 60}")
if errors:
    print(f"  \u2717 {len(errors)} layer(s) failed:")
    for desc, err in errors:
        print(f"    - {desc}: {err}")
empty = [f for f, c in results.items() if c == 0]
if empty:
    print(f"  \u26a0 {len(empty)} layer(s) returned 0 records:")
    for f in empty:
        print(f"    - {f}")
if total_records > 0:
    print(f"  \u2713 Total records: {total_records:,}")
    for f, c in results.items():
        if c > 0:
            print(f"    - {f}: {c:,}")
else:
    print(f"  \u2717 No records downloaded")
print(f"  Output: {OUTPUT_DIR}")
print("=" * 60)


## 📊 Verify Output & Schema Compatibility

In [ ]:
import os, json

TIGERWEB_SCHEMA = {
    "us_states.json": [
        "MTFCC","GEOID","STATE","STATENS","BASENAME","NAME","NAMELSAD",
        "LSADC","FUNCSTAT","AREALAND","AREAWATER","CENTLAT","CENTLON",
        "INTPTLAT","INTPTLON","OBJECTID"
    ],
    "us_counties.json": [
        "MTFCC","GEOID","STATE","COUNTY","BASENAME","NAME","NAMELSAD",
        "LSADC","FUNCSTAT","AREALAND","AREAWATER","CENTLAT","CENTLON",
        "INTPTLAT","INTPTLON","COUNTYNS","OBJECTID"
    ],
    "us_places.json": [
        "MTFCC","GEOID","STATE","PLACE","BASENAME","NAME","NAMELSAD",
        "LSADC","FUNCSTAT","AREALAND","AREAWATER","CENTLAT","CENTLON",
        "INTPTLAT","INTPTLON","PLACENS","OBJECTID"
    ],
    "us_county_subdivisions.json": [
        "MTFCC","GEOID","STATE","COUNTY","COUSUB","BASENAME","NAME",
        "NAMELSAD","LSADC","FUNCSTAT","AREALAND","AREAWATER","CENTLAT",
        "CENTLON","INTPTLAT","INTPTLON","COUSUBNS","OBJECTID"
    ],
    "us_mpos.json": [
        "OBJECTID","MPO_ID","MPO_NAME","NAME","STATE","AREA",
        "CENTLAT","CENTLON","INTPTLAT","INTPTLON","GEOID",
        "AREALAND","AREAWATER"
    ],
}

print("=== Schema Compatibility ===\n")
print(f"{'File':<35} {'Records':>8} {'Size':>8}  Schema")
print("-" * 75)

all_ok = True
for f in sorted(os.listdir(OUTPUT_DIR)):
    if not f.endswith(".json"):
        continue
    filepath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(filepath)
    with open(filepath) as fh:
        data = json.load(fh)
    count = data.get("_metadata", {}).get("recordCount", 0)
    records = data.get("records", [])
    size_str = f"{size / (1024*1024):.1f}MB" if size > 1024*1024 else f"{size / 1024:.0f}KB"

    if f in TIGERWEB_SCHEMA and records:
        actual_fields = set(records[0].keys())
        expected = set(TIGERWEB_SCHEMA[f])
        missing = expected - actual_fields
        if missing:
            schema_status = f"\u2717 missing: {missing}"
            all_ok = False
        else:
            schema_status = "\u2713 TIGERweb-compatible"
    else:
        schema_status = "?"

    status = "\u2713" if count > 0 else "\u2717"
    print(f"  {status} {f:<33} {count:>8} {size_str:>8}  {schema_status}")

# ── Backfill quality checks ──
print(f"\n{'='*60}")
print("=== Backfill Quality Checks ===")
print(f"{'='*60}")

# 1. States STATENS
if os.path.exists(os.path.join(OUTPUT_DIR, "us_states.json")):
    with open(os.path.join(OUTPUT_DIR, "us_states.json")) as fh:
        states = json.load(fh)["records"]
    filled = sum(1 for r in states if r.get('STATENS'))
    dc = next((r for r in states if r['GEOID'] == '11'), None)
    pr = next((r for r in states if r['GEOID'] == '72'), None)
    print(f"\n  States STATENS: {filled}/{len(states)} populated")
    if dc: print(f"  DC: LSADC={dc['LSADC']} STATENS={dc['STATENS']}")
    if pr: print(f"  PR: LSADC={pr['LSADC']} STATENS={pr['STATENS']}")

# 2. Counties LSADC
if os.path.exists(os.path.join(OUTPUT_DIR, "us_counties.json")):
    with open(os.path.join(OUTPUT_DIR, "us_counties.json")) as fh:
        counties = json.load(fh)["records"]
    lsadc_dist = {}
    for r in counties:
        lsadc_dist[r['LSADC']] = lsadc_dist.get(r['LSADC'], 0) + 1
    print(f"\n  Counties LSADC: {lsadc_dist}")
    va = [r for r in counties if r['GEOID'] == '51087']
    if va:
        print(f"  Henrico: {json.dumps(va[0], indent=2)}")

# 3. Places BASENAME & LSADC
if os.path.exists(os.path.join(OUTPUT_DIR, "us_places.json")):
    with open(os.path.join(OUTPUT_DIR, "us_places.json")) as fh:
        places = json.load(fh)["records"]
    # BASENAME should NOT contain suffixes
    bad = sum(1 for r in places if any(r['BASENAME'].endswith(s)
              for s in [' CDP',' city',' town',' village',' borough',
                        ' municipality',' comunidad',' zona urbana']))
    print(f"\n  Places BASENAME with leftover suffix: {bad}/{len(places)}")
    # LSADC distribution
    lsadc = {}
    for r in places:
        lsadc[r['LSADC']] = lsadc.get(r['LSADC'], 0) + 1
    print(f"  Places LSADC: {lsadc}")
    # Spot-check
    for search in ['Richmond city', 'Tuckahoe CDP']:
        match = [r for r in places if r['NAME'] == search]
        if match:
            r = match[0]
            print(f"  {r['NAME']}: BASENAME='{r['BASENAME']}' LSADC={r['LSADC']} MTFCC={r['MTFCC']}")

# 4. MPO centroids
if os.path.exists(os.path.join(OUTPUT_DIR, "us_mpos.json")):
    with open(os.path.join(OUTPUT_DIR, "us_mpos.json")) as fh:
        mpos = json.load(fh)["records"]
    with_c = sum(1 for r in mpos if r.get('CENTLAT'))
    print(f"\n  MPOs with centroids: {with_c}/{len(mpos)}")
    rico = [r for r in mpos if 'richmond' in r.get('MPO_NAME','').lower()]
    if rico:
        print(f"  {rico[0]['MPO_NAME']}: ({rico[0]['CENTLAT']}, {rico[0]['CENTLON']})")

print(f"\n{'='*60}")
if all_ok:
    print("\u2713 All layers TIGERweb schema-compatible")
else:
    print("\u26a0 Schema issues found — check above")


## 🔍 Quick Peek (Optional)

In [ ]:
#@title Peek at a downloaded file { display-mode: "form" }
FILE_TO_PEEK = "us_counties.json"  #@param ["us_states.json", "us_counties.json", "us_places.json", "us_county_subdivisions.json", "us_mpos.json"]
NUM_RECORDS = 3  #@param {type:"slider", min:1, max:20, step:1}

filepath = os.path.join(OUTPUT_DIR, FILE_TO_PEEK)
if os.path.exists(filepath):
    with open(filepath) as f:
        data = json.load(f)
    meta = data.get("_metadata", {})
    records = data.get("records", [])
    print(f"File: {FILE_TO_PEEK}")
    print(f"Source: {meta.get('dataSource', '?')}")
    print(f"Schema: {meta.get('schemaCompat', '?')}")
    print(f"Records: {meta.get('recordCount', '?')}")
    if records:
        print(f"Fields ({len(records[0])}): {list(records[0].keys())}")
        print(f"\nSample:\n")
        for r in records[:NUM_RECORDS]:
            print(json.dumps(r, indent=2))
            print()
else:
    print(f"Not found: {filepath}")


## 📥 Download as ZIP (if not using Drive)

In [ ]:
if not SAVE_TO_DRIVE:
    import shutil
    zip_path = shutil.make_archive("/content/geographic-data", "zip", OUTPUT_DIR)
    print(f"\u2713 Created: {zip_path}")
    from google.colab import files
    files.download(zip_path)
else:
    print("Files already saved to Google Drive")
    print(f"  Path: {OUTPUT_DIR}")
